**Purpose:** Manual labeling workflow for Reddit posts.

**Inputs:** `/kaggle/input/reddit-posts/reddit.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
# #model_name = "Qwen/Qwen2.5-7B-Instruct"                     # [315e0, 69797]
# #model_name = "google/gemma-7b-it"                           # [315e0]
# model_name = "meta-llama/Llama-3.1-8B-Instruct"             # [315e0, 9d5c9, 69797]

# !pip install -q transformers accelerate torch huggingface_hub pandas

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from huggingface_hub import login
# import torch
# import time
# import re
# import json
# import pandas as pd
# import hashlib
# import warnings
# warnings.filterwarnings("ignore")

# def recreat_post(row):
#     title = row["submission"]["title"]
#     selftext = row["submission"]["selftext"]

#     top_comments = [(com["body"], com["score"]) for com in row["top_comments"]]
#     top_comments = sorted(top_comments, key=lambda x: x[1], reverse=True)[:5]
#     top_comments = [com[0] for com in top_comments]

#     return {
#         "title": title,
#         "selftext": selftext,
#         "top_comments": top_comments
#     }

# def deterministic_shuffle(df):
#     keys = (
#         df.index.astype(str) + "_" +
#         df["subreddit"].astype(str) + "_" +
#         df["created_utc"].astype(str)
#     )
#     df = df.assign(
#         hash=keys.map(lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16))
#     )
#     return df.sort_values("hash").drop(columns="hash")

# login(token=HF_TOKEN)  # from src.secrets (see src/secrets_example.py)

# # --- Load model and tokenizer ---
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# # Load model on GPU automatically
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype="auto",
# )

# # --- Define prompt ---
# def label_word_sector(reddit_post, reddit_post_id, model, tokenizer, max_new_tokens=500):
#     prompt = """
# You are a financial domain classifier.

# Your task is to read Reddit stock discussion threads and determine which official **GICS sector(s)** the discussion most directly relates to.

# Use only these 11 GICS sectors:
# 1. Energy
# 2. Materials
# 3. Industrials
# 4. Consumer Discretionary
# 5. Consumer Staples
# 6. Health Care
# 7. Financials
# 8. Information Technology
# 9. Communication Services
# 10. Utilities
# 11. Real Estate

# Guidelines:
# - Base your answer on the companies, industries, or topics mentioned.
# - If multiple sectors are clearly relevant, include all of them.
# - If uncertain, pick the single most relevant one.
# - If no clear link exists, output "Unknown".
# - Provide a confidence score between 0 and 1 representing how certain you are of the classification.
# - Output strictly in JSON format.

# Expected output JSON:
# {{
#   "sector": ["<one or more GICS sectors>"],
#   "rationale": "<short explanation of why these sectors were chosen>",
#   "confidence": <float between 0 and 1>
# }}

# Classify the following Reddit thread:

# Title: """ + reddit_post['title'] + """

# Post:
# """ + reddit_post['selftext'] + """

# Top Comments:
# """ + ''.join(['- ' + comment + '\n' for comment in reddit_post['top_comments']])

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=0.0,
#             do_sample=False,
#             eos_token_id=tokenizer.eos_token_id,
#             repetition_penalty=1.0
#         )

#     text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     text_generated = text.split(prompt)[-1].strip()
#     text_generated = text_generated.strip().replace("{{", "{").replace("}}", "}")
#     match = re.search(r"\{.*?\}", text_generated, re.S)
#     if not match:
#         return {"post_id": reddit_post_id, "error": "no JSON"}
#     try:
#         return json.loads(match.group(0))
#     except Exception:
#         return {"post_id": reddit_post_id, "error": "invalid JSON"}
    
# # --- Load data ---
# df = pd.read_parquet("/kaggle/input/reddit-posts/reddit.parquet")
# df["top_comments"] = df["top_comments"].apply(json.loads)
# df["submission"] = df["submission"].apply(json.loads)

# df["has_selftext"] = df["submission"].apply(lambda x: "selftext" in x)
# df = df[df["has_selftext"]]
# df = deterministic_shuffle(df)

# # --- Results ---
# max_duration = 10 * 3600  # 10 hours
# start_time = time.time()

# records = []
# for i, row in df.iterrows():
#     elapsed_time = time.time() - start_time
#     if elapsed_time > max_duration:
#         print("⏰ Time limit reached (10 hours). Stopping the loop.")
#         break
    
#     reddit_post_id = i
#     reddit_post = recreat_post(row)

#     result = label_word_sector(reddit_post, reddit_post_id, model, tokenizer)

#     if "error" in result:
#         result = label_word_sector(reddit_post, reddit_post_id, model, tokenizer, max_new_tokens=1000)
    
#     records.append({
#         "reddit_post_id": reddit_post_id,
#         f"{model_name}": result
#     })
# df = pd.DataFrame(records)
# df.to_parquet(f"reddit_{model_name.split('/')[0]}.parquet", engine="pyarrow", index=False)